### Included Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

import os, random, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score


### Data Loading

In [2]:

IN = Path.cwd() / "data_frames"

train_df     = pd.read_parquet(IN / "train_df.parquet")
val_df       = pd.read_parquet(IN / "val_df.parquet")
train_df_nl  = pd.read_parquet(IN / "train_df_nl.parquet")
val_df_nl    = pd.read_parquet(IN / "val_df_nl.parquet")

# Arrays + feature names
arr = np.load(IN / "meta_arrays_v1.npz", allow_pickle=True)
Xtr_meta, Xva_meta = arr["Xtr"], arr["Xva"]
ytr, yva = arr["ytr"], arr["yva"]
kept_feature_cols = arr["feat_cols"].tolist()

In [3]:


# ---- Repro
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

# ---- Device (Apple GPU -> mps)
device = (
    torch.device("mps") if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
print("Device:", device)

# ---- Label mapping (from TRAIN ONLY, closed-set)
LABEL = "species" if "species" in train_df.columns else "category_id"
classes = pd.Index(sorted(train_df[LABEL].unique()))
cls2id  = {c:i for i,c in enumerate(classes)}
id2cls  = {i:c for c,i in cls2id.items()}

train_df = train_df.copy(); val_df = val_df.copy()
train_df["y"] = train_df[LABEL].map(cls2id).astype(int)
val_df["y"]   = val_df[LABEL].map(cls2id).astype(int)  # should be valid in closed-set

# (important for metadata alignment later)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# ---- Class weights (helps long tail; optional)
from collections import Counter
cnt = Counter(train_df["y"].tolist())
freq = np.array([cnt[i] for i in range(len(classes))], dtype=np.float32)
class_weights = torch.tensor((freq.sum()/(len(freq)*freq)), dtype=torch.float32).to(device)


Device: mps


### Dataset Loader

In [4]:
class DFImageDataset(Dataset):
    def __init__(self, df, img_col="image_path", y_col="y", transform=None):
        self.df = df.reset_index(drop=True)
        self.img_col, self.y_col, self.transform = img_col, y_col, transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row[self.img_col]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, int(row[self.y_col]), idx  # return idx for metadata lookup

IMG_SIZE = 300  # try 500 later for extra accuracy
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85,1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = DFImageDataset(train_df, transform=train_tfms)
val_ds   = DFImageDataset(val_df,   transform=val_tfms)

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


### Image only baseline

In [5]:
import timm

num_classes = len(classes)
model_img = timm.create_model("efficientnet_b3", pretrained=True, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)  # you can drop weight=... to compare
optimizer = torch.optim.AdamW(model_img.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)  # 10 epochs start

def run_epoch(model, loader, train=True):
    model.train(train)
    losses, yh, ph = [], [], []
    for imgs, y, _ in loader:
        imgs, y = imgs.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            logits = model(imgs)
            loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
        ph.append(logits.argmax(1).detach().cpu().numpy())
        yh.append(y.detach().cpu().numpy())
    y_true = np.concatenate(yh); y_pred = np.concatenate(ph)
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    return float(np.mean(losses)), acc, f1

best_f1 = -1
EPOCHS = 10
for ep in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_f1 = run_epoch(model_img, train_loader, train=True)
    va_loss, va_acc, va_f1 = run_epoch(model_img, val_loader,   train=False)
    scheduler.step()
    print(f"[ImgOnly] Ep{ep:02d} | tr {tr_loss:.3f}/{tr_acc:.3f}/{tr_f1:.3f} "
          f"| val {va_loss:.3f}/{va_acc:.3f}/{va_f1:.3f}")
    if va_f1 > best_f1:
        best_f1 = va_f1
        os.makedirs("runs", exist_ok=True)
        torch.save({"model": model_img.state_dict(), "classes": classes.tolist(), "img_size": IMG_SIZE},
                   "runs/effb3_imgonly_best.pt")
        print(f"  ↳ saved best (val macro-F1={best_f1:.3f})")


/Users/noshenatashe/Library/Mobile Documents/com~apple~CloudDocs/MIDS/DATASCI_207/fungitastic-classification-datasci207-Fall-2025/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/noshenatashe/Library/Mobile Documents/com~apple~CloudDocs/MIDS/DATASCI_207/fungitastic-classification-datasci207-Fall-2025/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/homebrew/Cellar/python@3.12/3.12.11_1/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
       

KeyboardInterrupt: 

In [ ]:
# Ensure shapes line up (same rows as train_df/val_df after your pruning order)
assert Xtr_meta.shape[0] == len(train_df)
assert Xva_meta.shape[0] == len(val_df)

# Backbone without classifier -> feature vector
backbone = timm.create_model("efficientnet_b3", pretrained=True, num_classes=0).to(device)
feat_dim = backbone.num_features
meta_dim = Xtr_meta.shape[1]
num_classes = len(classes)

class FusionModel(nn.Module):
    def __init__(self, backbone, feat_dim, meta_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(feat_dim + meta_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
    def forward(self, images, meta_batch):
        feats = self.backbone(images)            # [B, feat_dim]
        x = torch.cat([feats, meta_batch], 1)    # [B, feat_dim+meta_dim]
        return self.head(x)

model_fuse = FusionModel(backbone, feat_dim, meta_dim, num_classes).to(device)

crit_fuse = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
opt_fuse  = torch.optim.AdamW(model_fuse.parameters(), lr=1e-3, weight_decay=1e-4)
sch_fuse  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fuse, T_max=10)

def run_epoch_fusion(model, loader, X_meta, train=True):
    model.train(train)
    losses, yh, ph = [], [], []
    for imgs, y, idx in loader:
        imgs, y = imgs.to(device), y.to(device)
        meta_np = X_meta[idx.numpy()].astype(np.float32)
        meta = torch.from_numpy(meta_np).to(device)
        with torch.set_grad_enabled(train):
            logits = model(imgs, meta)
            loss = crit_fuse(logits, y)
        if train:
            opt_fuse.zero_grad(); loss.backward(); opt_fuse.step()
        losses.append(loss.item())
        ph.append(logits.argmax(1).detach().cpu().numpy())
        yh.append(y.detach().cpu().numpy())
    y_true = np.concatenate(yh); y_pred = np.concatenate(ph)
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    return float(np.mean(losses)), acc, f1

best_f1_fuse = -1
for ep in range(1, 11):
    tr_loss, tr_acc, tr_f1 = run_epoch_fusion(model_fuse, train_loader, Xtr_meta, train=True)
    va_loss, va_acc, va_f1 = run_epoch_fusion(model_fuse, val_loader,   Xva_meta, train=False)
    sch_fuse.step()
    print(f"[Fusion] Ep{ep:02d} | tr {tr_loss:.3f}/{tr_acc:.3f}/{tr_f1:.3f} "
          f"| val {va_loss:.3f}/{va_acc:.3f}/{va_f1:.3f}")
    if va_f1 > best_f1_fuse:
        best_f1_fuse = va_f1
        torch.save({"model": model_fuse.state_dict(), "classes": classes.tolist(),
                    "img_size": IMG_SIZE, "meta_dim": meta_dim},
                   "runs/effb3_fusion_best.pt")
        print(f"  ↳ saved best fusion (val macro-F1={best_f1_fuse:.3f})")
